In [1]:
import torch
import time
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

# 1. Base Model Preparation

In [2]:
MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32
)

model = model.to("cpu")
model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960, padding_idx=2)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=960, bias=False)
          (k_proj): Linear(in_features=960, out_features=320, bias=False)
          (v_proj): Linear(in_features=960, out_features=320, bias=False)
          (o_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-05)
    (r

In [3]:
messages = [
    {"role": "user", "content": "Write a short friendly welcome message for a fine-tuning workshop."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
Write a short friendly welcome message for a fine-tuning workshop.
assistant
Welcome to the Fine-Tuning Workshop! I'm thrilled to be your guide and friend as we embark on this exciting journey to fine-tune your models and unlock their full potential. We'll explore the latest techniques and tools, and I'll share my expertise to help you achieve the best results.

As we work together, I'll provide you with practical tips and tricks to improve your models


In [4]:
outputs.shape

torch.Size([1, 123])

In [5]:
def benchmark_generate(model, tokenizer, messages, max_new_tokens=80, do_sample=False):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    input_ids = inputs["input_ids"]
    input_len = input_ids.shape[1]

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )

    end = time.perf_counter()

    generated_ids = outputs[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    total_time_s = end - start
    generated_tokens = generated_ids.shape[0]
    tokens_per_second = generated_tokens / total_time_s if total_time_s > 0 else 0.0

    return {
        "text": generated_text,
        "total_time_s": total_time_s,
        "generated_tokens": generated_tokens,
        "tokens_per_second": tokens_per_second,
    }

In [6]:
result = benchmark_generate(
    model,
    tokenizer,
    messages,
    max_new_tokens=80,
    do_sample=False
)

In [7]:
print(result["text"])
print()
print(np.round(result["total_time_s"],2))
print(np.round(result["generated_tokens"],2))
print(np.round(result["tokens_per_second"],2))

Welcome to the Fine-Tuning Workshop! I'm thrilled to be your guide and friend as we embark on this exciting journey to fine-tune your models and unlock their full potential. We'll explore the latest techniques and tools, and I'll share my expertise to help you achieve the best results.

As we work together, I'll provide you with practical tips and tricks to improve your models

7.55
80
10.6


In [8]:
result

{'text': "Welcome to the Fine-Tuning Workshop! I'm thrilled to be your guide and friend as we embark on this exciting journey to fine-tune your models and unlock their full potential. We'll explore the latest techniques and tools, and I'll share my expertise to help you achieve the best results.\n\nAs we work together, I'll provide you with practical tips and tricks to improve your models",
 'total_time_s': 7.549074699985795,
 'generated_tokens': 80,
 'tokens_per_second': 10.597325258968564}

In [9]:
messages = [
    {"role": "user", "content": "How to teach my dog how to sit"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
How to teach my dog how to sit
assistant
Sure, here's a step-by-step guide on how to teach your dog to sit:

1. **Choose a Quiet Place**: Find a quiet, comfortable place where your dog can relax and focus. This could be a room with a door that's closed, or a room with a window that's closed.

2. **Prepare the Treat**: Make sure you have a treat that your dog will find and enjoy. It's best to use a small, easy-to-handle treat.

3. **Position Your Dog**: Stand in front of your dog, with your back to them. Make sure you're facing the same direction.

4. **Hold the Treat**: Hold the treat in your hand, palm facing up. This will help your dog associate the treat with the action of sitting.

5. **Position Your Dog**: Slowly and calmly, place the treat


In [10]:
messages = [
    {"role": "user", "content": "How to teach my dog how to stay, return JSON"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
How to teach my dog how to stay, return JSON
assistant
To teach your dog how to stay, return JSON, you can follow these steps:

1. Start by placing your dog in a safe and quiet area.

2. Start by having your dog sit. This is a basic command that your dog will learn quickly.

3. Have your dog sit and then have you walk away from them.

4. Have your dog stay in the same position as they were in when you first placed them in the safe area.

5. Repeat this process several times until your dog starts to associate staying in the safe area with the command "stay".

6. Once your dog is comfortable with the command "stay", you can start to move them around the room.

7. When you move your dog, have them return to the safe area.

8. Repeat this process several times until your dog starts to return to the safe area when


# 2. LoRA Finetune 

In [11]:
from pathlib import Path

In [12]:
curr_dir = Path.cwd()
curr_dir.is_dir()

True

In [83]:
output_dir = curr_dir.parent/"outputs"/"models"
output_dir.is_dir()

True

In [13]:
[item for item in curr_dir.parent.iterdir()]

[WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/.git'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/.gitignore'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/.ipynb_checkpoints'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/.python-version'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/.venv'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/.venv_arm64'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/.venv_x86'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/configs'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/datasets'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/LICENSE'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops/model-alignment-lab/main.py'),
 WindowsPath('C:/Users/DFS/Desktop/gitrepo/workshops

In [14]:
datasets_dir = curr_dir.parent/"datasets"/"structured_json"
datasets_dir.is_dir()

True

In [15]:
train_path = datasets_dir.joinpath("TRAIN_dog_training_100_samples.jsonl")
train_path.is_file()

True

In [16]:
test_path = datasets_dir.joinpath("TEST_dog_training_test_20_samples.jsonl")
test_path.is_file()

True

## Load the Dataset

In [17]:
from datasets import load_dataset

train_dataset = load_dataset("json", 
                             data_files={
                                 "train": str(train_path)
                             }
                            )
test_dataset = load_dataset("json",
                            data_files={
                                "test": str(test_path)
                            }
                           )

In [38]:
split_dataset = train_dataset["train"].train_test_split(
    test_size=0.2,
    seed=41
)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

In [45]:
split_dataset

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 80
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 20
    })
})

In [40]:
train_dataset

Dataset({
    features: ['messages'],
    num_rows: 80
})

In [41]:
val_dataset

Dataset({
    features: ['messages'],
    num_rows: 20
})

In [46]:
test_dataset = test_dataset["test"]
test_dataset

Dataset({
    features: ['messages'],
    num_rows: 20
})

In [47]:
train_dataset["messages"][1]

[{'role': 'user',
  'content': 'I’m training at home. How do I get my dog to sit on command?'},
 {'role': 'assistant',
  'content': '{"command": "sit", "difficulty": "beginner", "steps": ["Reward immediately when seated", "Move the treat upward so the head follows", "Repeat in short sessions", "Wait for the dog\'s bottom to lower"], "common_mistake": "Using a treat lure that moves too fast", "session_length_minutes": 4, "reward_tip": "Use soft treats for quick rewards"}'}]

In [48]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}
    
train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)
test_dataset = test_dataset.map(format_example)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [50]:
train_dataset

Dataset({
    features: ['messages', 'text'],
    num_rows: 80
})

In [53]:
train_dataset["text"][0]

'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nI’m working with a young dog. How should I start teaching watch me? I want beginner-friendly steps.<|im_end|>\n<|im_start|>assistant\n{"command": "watch me", "difficulty": "beginner", "steps": ["Build duration slowly over time", "Say the cue watch me", "Hold a treat near your eyes", "Reward immediately for attention"], "common_mistake": "Repeating the cue constantly", "session_length_minutes": 4, "reward_tip": "Practice in very short bursts"}<|im_end|>\n'

### LoRA Setup

In [54]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)


In [74]:
model = model.unload()
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 960, padding_idx=2)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): Linear(in_features=960, out_features=960, bias=False)
              (k_proj): Linear(in_features=960, out_features=320, bias=False)
              (v_proj): Linear(in_features=960, out_features=320, bias=False)
              (o_proj): Linear(in_features=960, out_features=960, bias=False)
            )
            (mlp): LlamaMLP(
              (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
              (up_proj): Linear(in_features=960, out_features=2560, bias=False)
              (down_proj): Linear(in_features=2560, out_features=960, bias=False)
              (act_fn): SiLUActivation()
            )
            (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
  

In [75]:
model = get_peft_model(model, peft_config).to("cpu")
model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): LlamaForCausalLM(
          (model): LlamaModel(
            (embed_tokens): Embedding(49152, 960, padding_idx=2)
            (layers): ModuleList(
              (0-31): 32 x LlamaDecoderLayer(
                (self_attn): LlamaAttention(
                  (q_proj): lora.Linear(
                    (base_layer): Linear(in_features=960, out_features=960, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=960, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=960, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [77]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): LlamaForCausalLM(
          (model): LlamaModel(
            (embed_tokens): Embedding(49152, 960, padding_idx=2)
            (layers): ModuleList(
              (0-31): 32 x LlamaDecoderLayer(
                (self_attn): LlamaAttention(
                  (q_proj): lora.Linear(
                    (base_layer): Linear(in_features=960, out_features=960, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=960, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=960, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [70]:
model.print_trainable_parameters()

trainable params: 1,638,400 || all params: 363,459,520 || trainable%: 0.4508


## Training

In [72]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(output_dir/"smollm-dog-lora"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="text",
    use_cpu=True,
    bf16=False,
    fp16=False,
    assistant_only_loss=True
)

In [73]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

In [78]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.629791,2.556897
2,2.115139,2.093478
3,1.861013,1.928790


TrainOutput(global_step=60, training_loss=2.3393096923828125, metrics={'train_runtime': 856.7236, 'train_samples_per_second': 0.28, 'train_steps_per_second': 0.07, 'total_flos': 64233270518400.0, 'train_loss': 2.3393096923828125})

In [81]:
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
ts

'20260401_143510'

In [92]:
final_path = str(output_dir/"smollm-dog-lora")
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter")

('C:\\Users\\DFS\\Desktop\\gitrepo\\workshops\\model-alignment-lab\\outputs\\models\\smollm-dog-lora/20260401_143510_final_adapter\\tokenizer_config.json',
 'C:\\Users\\DFS\\Desktop\\gitrepo\\workshops\\model-alignment-lab\\outputs\\models\\smollm-dog-lora/20260401_143510_final_adapter\\chat_template.jinja',
 'C:\\Users\\DFS\\Desktop\\gitrepo\\workshops\\model-alignment-lab\\outputs\\models\\smollm-dog-lora/20260401_143510_final_adapter\\tokenizer.json')

In [93]:
def generate_response(model, tokenizer, user_prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": user_prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [94]:
for i in range(5):
    sample = test_dataset[i]
    prompt = sample["messages"][0]["content"]
    reference = sample["messages"][1]["content"]

    pred = generate_response(model, tokenizer, prompt)

    print("=" * 80)
    print("PROMPT:")
    print(prompt)
    print("\nPREDICTION:")
    print(pred)
    print("\nREFERENCE:")
    print(reference)
    print()

PROMPT:
I'm training indoors. Can you show me how to teach stay? I want simple steps.

PREDICTION:
Sure, I'd be happy to help you train your dog to stay indoors. Here are the steps:

1. **Choose a Quiet Place**: Find a quiet, private area where your dog can stay calm. This could be a room with a door that's closed, or a room with a window that's closed.

2. **Start with a Short Stay**: Begin with a short stay of 1-2 minutes. This will help your dog understand that staying is a command.

3. **Gradually Increase Duration**: Once your dog is comfortable with the short stay, gradually increase the duration. For example, you could start with 2 minutes and then increase to 5 minutes.

4. **Use a Treat**: Use a treat to lure your dog into the room. This will help them associate the stay command with the reward.

5. **Keep the Command Simple**: Keep the command simple. You can say "stay" and then give your dog a treat

REFERENCE:
{"command": "stay", "difficulty": "intermediate", "steps": ["Inc

In [96]:
prompt = """How do I teach my dog to stay?
Respond in JSON only using this schema:
command, difficulty, steps, common_mistake, session_length_minutes, reward_tip."""

messages = [
    {"role": "user", "content": prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
How do I teach my dog to stay?
Respond in JSON only using this schema:
command, difficulty, steps, common_mistake, session_length_minutes, reward_tip.
assistant
{
  "command": "stay",
  "difficulty": "easy",
  "steps": [
    {
      "difficulty": "medium",
      "steps": [
        "Hold a treat close to your dog's nose and say "stay" in a calm, clear voice.",
        "Bring the treat closer to your dog's head and say "stay" again.",
        "Keep


In [95]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): LlamaForCausalLM(
          (model): LlamaModel(
            (embed_tokens): Embedding(49152, 960, padding_idx=2)
            (layers): ModuleList(
              (0-31): 32 x LlamaDecoderLayer(
                (self_attn): LlamaAttention(
                  (q_proj): lora.Linear(
                    (base_layer): Linear(in_features=960, out_features=960, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=960, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=960, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            